In [ ]:
# Naive Bayes is a ML algoritthm that uses probability(Bayes Theorem) to make predictions.
# It is a classification algorithm that assumes that the features are independent of each other. It is called "naive" because of this assumption, which is often not true in real-world data. Despite this, Naive Bayes can perform surprisingly well in many applications, especially in text classification and spam detection.
# It combines` prior knowledge (prior probabilities)` with the `likelihood` of the observed data to make predictions about the class of new instances. 

# Naive Bayes is a machine learning algorithm that uses probability to make predictions.
# It looks at the evidence, calculates how likely something is to belong to a particular category, and chooses the category with the highest probability.

In [ ]:
# A REALLY DUMB SPAM FILTER

# We have two events:
# - S = The message is spam.
# - B = The message contains the word "bitcoin".
#
# What is the probability that a message is spam, GIVEN that it contains the word bitcoin?
#
# The formula is:
#                 P(B|S)P(S)
# P(S|B) = -----------------------
#          P(B|S)P(S)+P(B|¬S)P(¬S)
#
# - Numerator is the probability that a message is spam and contains bitcoin
# - Denominator is the probability that a message contains bitcoin, whether it is spam or not.
# - Hence, think of this calculation as representing the proportion of bitcoin messages that are spam.
# 

In [ ]:
# A MORE SOPHISTICATED SPAM FILTER

# - We have a vocabulary of words, w1, w2, ..., wn.
# - Xi is the event that the message contains the word wi.
# - P(Xi|S) is the probability that a spam message contains the ith word.
# - P(Xi|¬S) is the probability that a non-spam message contains the ith word.

# - The key to Naive Bayes is to make the (big) assumption that the words are independent of each other. This allows us to calculate the probability of a message being spam given the words it contains as follows:
# P(X1 = x1, ..., Xn = xn|S) = P(X1 = x1|S) * ... * P(Xn = xn|S)

# Can calculate the probability of a message being spam given the words it contains as follows:
# P(S|X=x) = P(X=x|S) / [P(X=x|S) + P(X=x|¬S)]

# The Naive Bayes assumption allows us to compute each of the pribabilities on the right simpley by multiplying the probabilities of the individual words, which is much easier than trying to compute the joint probability of all the words together.
#

# In procatise, we want to avoid multiplying a lot of small probabilities together, which can lead to numerical underflow. Instead, we can use logarithms to convert the products into sums, which are easier to compute and less prone to numerical issues.

# Recall;
# log(a * b) = log(a) + log(b)
# exp(log(a) + log(b)) = exp(log(a)) * exp(log(b)) = a * b

# Hence we compute, p1 * ... * pn as exp(log(p1) + ... + log(pn)).

# If in our training dataset the word 'data' only accurs iin nonspam messages, then P(Xi|S) = 0. This is a problem because it means that any message containing the word 'data' will have a probability of being spam of 0, regardless of the other words in the message. To avoid this, we can use Laplace smoothing, which adds a small constant (usually 1) to the count of each word in each class. This ensures that no probability is ever exactly 0.
# We'll use the pseodocount -k- and estimate the probability of seing the ith word in a spam message as:
# P(Xi|S) = (count(Xi in spam) + k) / (count(spam) + k * |V|), where |V| is the size of the vocabulary. Similarly, we can estimate the probability of seeing the ith word in a non-spam message as:
# P(Xi|¬S) = (count(Xi in non-spam) + k) / (count(non-spam) + k * |V|)

# Example;
# If data occurs in 0/98 spam messages and 10/902 non-spam messages, then with k=1, we have:
# P(Xi|S) = (0 + 1) / (98 + 1 * 1000) = 1 / 1098
# P(Xi|¬S) = (10 + 1) / (902 + 1 * 1000) = 11 / 1902

In [1]:
# IMPLEMENTATION
from typing import Set
import re

## Tokenize messages in disticnt words

def tokenize(text: str) -> Set[str]:
    text = text.lower()                         # Convert to lowercase
    all_words = re.findall("[a-z0-9]+", text)   # extract the words, and
    return set(all_words)                       # remove the duplicates

assert tokenize("Data is the new oil. Data is the new oil!") == {"data", "is", "the", "new", "oil"}

In [2]:
## Type for training data
from typing import NamedTuple

class Message(NamedTuple):
    text: str
    is_spam: bool

In [ ]:
# Classifier

from typing import List, Dict, Tuple, Iterable
import math
from collections import defaultdict

class NaiveBayesClassifier:
    def __init__(self, k: float = 0.5) -> None:
        self.k = k
        self.tokens: Set[str] = set()
        self.token_spam_counts: Dict[str, int] = defaultdict(int)
        self.token_ham_counts: Dict[str, int] = defaultdict(int)
        self.spam_messages = self.ham_messages = 0
        
    def train(self, messages: Iterable[Message]) -> None:
        """Captures the counts of spam and ham messages, as well as the counts of each token in spam and ham messages."""
        for message in messages:
            # Increment message counts
            if message.is_spam:
                self.spam_messages += 1
            else:
                self.ham_messages += 1

            # Increment word counts
            for token in tokenize(message.text):
                self.tokens.add(token)
                if message.is_spam:
                    self.token_spam_counts[token] += 1
                else:
                    self.token_ham_counts[token] += 1

    def _probabilities(self, token: str) -> Tuple[float, float]:
        """Returns the probabilities P(token|spam) and P(token|ham)"""
        spam_count = self.token_spam_counts[token]
        ham_count = self.token_ham_counts[token]

        p_token_given_spam = (spam_count + self.k) / (self.spam_messages + 2 * self.k)
        p_token_given_ham = (ham_count + self.k) / (self.ham_messages + 2 * self.k)

        return p_token_given_spam, p_token_given_ham

    def predict(self, text: str) -> float:
        """Returns the probability that the given message is spam."""
        tokens = tokenize(text)
        log_prob_if_spam = log_prob_if_ham = 0.0

        # Iterate through each word in our vocabulary
        for token in self.tokens:
            p_token_given_spam, p_token_given_ham = self._probabilities(token)

            # If "token" appears in message
            # add the log probability of seeing it
            if token in tokens:
                log_prob_if_spam += math.log(p_token_given_spam)
                log_prob_if_ham += math.log(p_token_given_ham)
            # Otherwise add the log probability of not seeing it
            # which is log(1 - probability of seeing it)
            else:
                log_prob_if_spam += math.log(1.0 - p_token_given_spam)
                log_prob_if_ham += math.log(1.0 - p_token_given_ham)

        prob_if_spam = math.exp(log_prob_if_spam)
        prob_if_ham = math.exp(log_prob_if_ham)

        return prob_if_spam / (prob_if_spam + prob_if_ham)

In [8]:
# TESTING THE MODEL

messages = [
    Message("spam rules", is_spam=True),
    Message("ham rules", is_spam=False),
    Message("hello ham", is_spam=False)
]


classifier = NaiveBayesClassifier()
classifier.train(messages)

assert classifier.tokens == {"spam", "rules", "ham", "hello"}
assert classifier.spam_messages == 1
assert classifier.ham_messages == 2
assert classifier.token_spam_counts == {"spam": 1, "rules": 1}
assert classifier.token_ham_counts == {"ham": 2, "hello": 1, "rules": 1}

In [10]:
# Make prdicition by hand

text = "hello spam"

probs_if_spam = [
    (1 + 0.5) / (1 + 2 * 0.5),      # "spam" (present)
    1 - (0 + 0.5) / (1 + 2 * 0.5),  # "ham" ( not present)
    1 - (1 + 0.5) / (1 + 2 * 0.5),  # "rules" (not present)
    (0 + 0.5) / (1 + 2 * 0.5)       # "hello" (present)
]

probs_if_ham = [
    (0 + 0.5) / (2 + 2 * 0.5),      # "spam" (present)
    1 - (2 + 0.5) / (2 + 2 * 0.5),  # "ham" (not present)
    1 - (1 + 0.5) / (2 + 2 * 0.5),  # "rules" (not present)
    (1 + 0.5) / (2 + 2 * 0.5)       # "hello" (present)
]

p_if_spam = math.exp(sum(math.log(p) for p in probs_if_spam))
p_if_ham = math.exp(sum(math.log(p) for p in probs_if_ham))

print(f"p_if_spam: {p_if_spam}, p_if_ham: {p_if_ham}")
print(f"p_spam_given_message: {p_if_spam / (p_if_spam + p_if_ham)}")

# Should be about 0.83
assert classifier.predict(text) == p_if_spam / (p_if_spam + p_if_ham)

p_if_spam: 0.03515625, p_if_ham: 0.006944444444444444
p_spam_given_message: 0.8350515463917525


In [12]:
# Using the model

from io import BytesIO  # So we can treat the downloaded bytes as a file
import requests         # To download the dataset, which
import tarfile          # are in .tar.bz

BASE_URL = "https://spamassassin.apache.org/old/publiccorpus/"
FILES = [
    "20021010_easy_ham.tar.bz2", 
    "20021010_hard_ham.tar.bz2", 
    "20021010_spam.tar.bz2"
]

OUTPUT_DIR = 'spam_data'

for filename in FILES:
    # Download the file
    print(f"Downloading {filename}...")
    response = requests.get(BASE_URL + filename)
    response.raise_for_status()  # Check that the download was successful

    # Open the tar.bz2 file
    with tarfile.open(fileobj=BytesIO(response.content), mode="r:bz2") as tar:
        # Extract all files to the output directory
        tar.extractall(path=OUTPUT_DIR)

In [19]:
# Get the list of all files in the spam_data directory 

import glob, os, re

files = glob.glob("spam_data/*/*")

data: List[Message] = []

for file in files:
    # Folder name is easy_ham, hard_ham, or spam (the path itself always contains "spam_data")
    is_spam = os.path.basename(os.path.dirname(file)) == "spam"
    with open(file, errors='ignore') as email_file:
        for line in email_file:
            # Only consider lines that start with "Subject:"
            if line.startswith("Subject: "):
                subject = line.lstrip("Subject: ")
                data.append(Message(subject, is_spam))
                break # done with this file

In [20]:
# Build Data into Training and Testing Sets

import sys
sys.path.append("..")  # make DS/ importable so `scratch` is found

import random
from scratch.machine_learning import split_data

random.seed(0)
train_data, test_data = split_data(data, 0.75)

model =NaiveBayesClassifier()
model.train(train_data)

In [21]:
# Make predictions on the test set and evaluate the model's accuracy

from collections import Counter

predictions = [
    (message, model.predict(message.text)) 
    for message in test_data
]

# Assume that spam_probability > 0.5 corresponds to spam prediction,
# and count the combinations of (actual is_spam, predicted is_spam)
confusion_matrix = Counter((message.is_spam, spam_probability > 0.5) for message, spam_probability in predictions)

print(confusion_matrix)


Counter({(False, False): 656, (True, True): 93, (True, False): 46, (False, True): 30})


In [ ]:
def p_spam_given_token(token: str, model: NaiveBayesClassifier) -> float:
    """Returns the probability that a message is spam given that it contains the given token."""
    probs_if_spam, probs_if_ham = model._probabilities(token)
    return probs_if_spam / (probs_if_spam + probs_if_ham)

words = sorted(model.tokens, key=lambda token: p_spam_given_token(token, model), reverse=True)

print("spammiest_words", words[-10:])
print("hammiest_words", words[:10])

spammiest words ['selling', 'wedded', 'bliss', 'apt', 'ouch', 'sadev', 'zzzzteana', 'razor', 'users', 'spambayes']
hammiest words ['adv', 'money', 'systemworks', 'year', 'sale', 'account', 'mortgage', '95', 'clearance', 'per']
